# Startup Funding Clustering Workflow

This notebook is the reproducible modeling workflow for the startup funding clustering project. It loads the raw data, documents cleaning decisions, creates funding-composition features, selects and validates K-Means clusters, and writes the final real-value profile dataset used by the interpretation notebook.

## Method Summary

- The final model fits K-Means on the full robust-scaled engineered feature set.
- PCA is used for visualization only, not as the final clustering input.
- The default model uses four clusters, with inertia, sampled silhouette, cluster sizes, and stability checks reported for context.
- Funding columns are highly skewed and sparse, so log transforms, composition ratios, and RobustScaler are used before clustering.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.3f}".format)

RAW_DATA = Path("investments_VC.csv")
OUTPUT_DATA = Path("clustered_startups_real_values.csv")
RANDOM_STATE = 42
OPTIMAL_K = 4
SILHOUETTE_SAMPLE_SIZE = 5000

## Load Data

Column names and text values are stripped immediately because the raw CSV contains whitespace around fields such as market and funding_total_usd.

In [ ]:
row_log = []

df_raw = pd.read_csv(RAW_DATA, encoding="latin1")
row_log.append({"step": "Raw file", "rows": len(df_raw), "columns": df_raw.shape[1]})

df = df_raw.copy()
df.columns = df.columns.str.strip()

text_columns = df.select_dtypes(include="object").columns
for col in text_columns:
    df[col] = df[col].astype("string").str.strip()
    df.loc[df[col].eq(""), col] = pd.NA

row_log.append({"step": "Column/text whitespace stripped", "rows": len(df), "columns": df.shape[1]})
display(pd.DataFrame(row_log))
display(df.head())

## Initial Data Quality

Before removing rows, inspect missingness and the raw market/status distributions. These checks give context for later filtering decisions.

In [ ]:
missing_before = (
    df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_pct")
    .to_frame()
)

print("Top missing-value rates before cleaning")
display(missing_before.head(15))

if "market" in df.columns:
    print("Top markets before cleaning")
    display(df["market"].value_counts(dropna=False).head(10))

if "status" in df.columns:
    print("Status distribution before cleaning")
    display(df["status"].value_counts(dropna=False))

## Numeric Cleaning And Suspicious Values

Funding totals are parsed from strings such as 17,50,000. A dash is treated as missing. Subtype and round columns are filled with zero after suspicious values are counted.

In [ ]:
irrelevant_cols = [
    "permalink",
    "homepage_url",
    "state_code",
    "region",
    "founded_month",
    "founded_quarter",
    "founded_year",
]
df = df.drop(columns=[col for col in irrelevant_cols if col in df.columns])
row_log.append({"step": "Irrelevant descriptor columns removed", "rows": len(df), "columns": df.shape[1]})

funding_type_cols = [
    "seed",
    "venture",
    "equity_crowdfunding",
    "undisclosed",
    "convertible_note",
    "debt_financing",
    "angel",
    "grant",
    "private_equity",
    "post_ipo_equity",
    "post_ipo_debt",
    "secondary_market",
    "product_crowdfunding",
]
round_cols = ["round_A", "round_B", "round_C", "round_D", "round_E", "round_F", "round_G", "round_H"]
numeric_cols = ["funding_total_usd", "funding_rounds", *funding_type_cols, *round_cols]
numeric_cols = [col for col in numeric_cols if col in df.columns]


def parse_numeric(series):
    cleaned = (
        series.astype("string")
        .str.replace(",", "", regex=False)
        .str.replace(r"^\s*-\s*$", "", regex=True)
        .str.strip()
    )
    return pd.to_numeric(cleaned, errors="coerce")


for col in numeric_cols:
    df[col] = parse_numeric(df[col])

funding_type_cols = [col for col in funding_type_cols if col in df.columns]
round_cols = [col for col in round_cols if col in df.columns]
subtype_total = df[funding_type_cols].fillna(0).sum(axis=1)

quality_checks = pd.DataFrame(
    {
        "check": [
            "negative numeric values",
            "negative funding_total_usd",
            "zero total funding with positive subtype funding",
            "subtype funding total greater than funding_total_usd",
        ],
        "rows": [
            int(df[numeric_cols].lt(0).any(axis=1).sum()),
            int(df["funding_total_usd"].lt(0).sum()) if "funding_total_usd" in df.columns else 0,
            int(((df["funding_total_usd"].fillna(0) == 0) & (subtype_total > 0)).sum()),
            int(((df["funding_total_usd"].fillna(0) > 0) & (subtype_total > df["funding_total_usd"].fillna(0))).sum()),
        ],
    }
)
display(quality_checks)

non_total_numeric = [col for col in numeric_cols if col != "funding_total_usd"]
df[non_total_numeric] = df[non_total_numeric].fillna(0).clip(lower=0)
df["funding_total_usd"] = df["funding_total_usd"].where(df["funding_total_usd"] >= 0)

if "status" in df.columns:
    df["status"] = df["status"].astype("category")

## Row Filtering Log

The row log keeps the academic audit trail visible: each major filtering step reports the number of remaining rows and columns.

In [ ]:
rows_before = len(df)
df = df.dropna(how="all")
row_log.append({"step": "Completely empty rows removed", "rows": len(df), "columns": df.shape[1], "removed": rows_before - len(df)})

rows_before = len(df)
df = df.dropna(subset=["name"])
row_log.append({"step": "Rows without company name removed", "rows": len(df), "columns": df.shape[1], "removed": rows_before - len(df)})

rows_before = len(df)
df = df.drop_duplicates()
row_log.append({"step": "Exact duplicate rows removed", "rows": len(df), "columns": df.shape[1], "removed": rows_before - len(df)})

rows_before = len(df)
df = df.dropna(subset=["funding_total_usd", "market"])
row_log.append({"step": "Rows without funding_total_usd or market removed", "rows": len(df), "columns": df.shape[1], "removed": rows_before - len(df)})

missing_after = (
    df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_pct")
    .to_frame()
)

display(pd.DataFrame(row_log))
print("Top missing-value rates after cleaning")
display(missing_after.head(15))

## Feature Engineering

The model uses real funding amounts, log-transformed amounts, funding-composition ratios, funding rounds, and an inferred stage_level from the highest nonzero round column. Seed, angel, and grant are combined into early_stage_funding to reduce sparsity.

In [ ]:
stage_order = {
    "round_A": 1,
    "round_B": 2,
    "round_C": 3,
    "round_D": 4,
    "round_E": 5,
    "round_F": 6,
    "round_G": 7,
    "round_H": 8,
}

df["stage_level"] = 0
for col, level in stage_order.items():
    if col in df.columns:
        df["stage_level"] = np.where(
            df[col].fillna(0) > 0,
            np.maximum(df["stage_level"], level),
            df["stage_level"],
        )

early_stage_components = [col for col in ["seed", "angel", "grant"] if col in df.columns]
df["early_stage_funding"] = df[early_stage_components].sum(axis=1) if early_stage_components else 0

core_funding_cols = ["early_stage_funding", "venture", "debt_financing", "private_equity"]
total_funding_safe = df["funding_total_usd"].replace(0, np.nan)

for col in core_funding_cols:
    df[f"{col}_ratio"] = df[col] / total_funding_safe

ratio_cols = [f"{col}_ratio" for col in core_funding_cols]
ratio_rows_above_one = int(df[ratio_cols].gt(1).any(axis=1).sum())
ratio_cells_above_one = int(df[ratio_cols].gt(1).sum().sum())

df[ratio_cols] = df[ratio_cols].fillna(0).clip(lower=0, upper=1)

columns_to_log = [
    "funding_total_usd",
    "funding_rounds",
    "early_stage_funding",
    "venture",
    "debt_financing",
    "private_equity",
]
for col in columns_to_log:
    df[f"log_{col}"] = np.log1p(df[col].clip(lower=0))

ratio_quality = pd.DataFrame(
    {
        "check": ["rows with any ratio above 1 before clipping", "ratio cells above 1 before clipping"],
        "count": [ratio_rows_above_one, ratio_cells_above_one],
    }
)
display(ratio_quality)

display(df[["name", "market", "funding_total_usd", "stage_level", *core_funding_cols, *ratio_cols]].head())

## Final Model Features

The table below documents why each feature is included. The clustering input is intentionally numeric and scaled, while company descriptors are kept only for interpretation and output.

In [ ]:
feature_reasons = pd.DataFrame(
    [
        ("log_funding_total_usd", "Overall funding scale after reducing skew"),
        ("log_funding_rounds", "Intensity and repetition of fundraising activity"),
        ("log_early_stage_funding", "Seed, angel, and grant funding as early capital"),
        ("log_venture", "Venture-backed funding magnitude"),
        ("log_debt_financing", "Debt-financing magnitude"),
        ("log_private_equity", "Private-equity funding magnitude"),
        ("stage_level", "Highest observed named round from A through H"),
        ("early_stage_funding_ratio", "Share of total funding from early-stage sources"),
        ("venture_ratio", "Share of total funding from venture capital"),
        ("debt_financing_ratio", "Share of total funding from debt"),
        ("private_equity_ratio", "Share of total funding from private equity"),
    ],
    columns=["feature", "reason"],
)

features_to_scale = feature_reasons["feature"].tolist()
display(feature_reasons)

## Scale The Clustering Frame

RobustScaler is used because funding distributions contain very large outliers. Scaling is fit only on the clustering-ready rows, and indexes are kept aligned for safe output assignment.

In [ ]:
df_clustering = df.dropna(subset=features_to_scale).copy()
row_log.append({"step": "Clustering-ready rows", "rows": len(df_clustering), "columns": df_clustering.shape[1], "removed": len(df) - len(df_clustering)})

scaler = RobustScaler()
scaled_array = scaler.fit_transform(df_clustering[features_to_scale])
df_scaled = pd.DataFrame(scaled_array, columns=features_to_scale, index=df_clustering.index)

assert df_scaled.index.equals(df_clustering.index)
assert not df_scaled.isna().any().any()

display(pd.DataFrame(row_log))
display(df_scaled.describe().T)

## Select The Number Of Clusters

The silhouette score is estimated on a fixed sample to keep the notebook practical on this dataset size. Inertia, silhouette, and cluster-size balance are reported for k from 2 to 9.

In [ ]:
model_selection_rows = []

for k in range(2, 10):
    kmeans = KMeans(n_clusters=k, init="k-means++", random_state=RANDOM_STATE, n_init=10)
    labels = kmeans.fit_predict(df_scaled)
    counts = pd.Series(labels).value_counts()
    sample_size = min(SILHOUETTE_SAMPLE_SIZE, len(df_scaled))
    score = silhouette_score(df_scaled, labels, sample_size=sample_size, random_state=RANDOM_STATE)
    model_selection_rows.append(
        {
            "k": k,
            "inertia": kmeans.inertia_,
            "sampled_silhouette": score,
            "min_cluster_size": int(counts.min()),
            "max_cluster_size": int(counts.max()),
            "smallest_cluster_pct": counts.min() / len(labels) * 100,
        }
    )

model_selection_df = pd.DataFrame(model_selection_rows)
display(model_selection_df)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(model_selection_df["k"], model_selection_df["inertia"], marker="o", color="tab:blue")
ax1.set_xlabel("Number of clusters (k)")
ax1.set_ylabel("Inertia", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.plot(model_selection_df["k"], model_selection_df["sampled_silhouette"], marker="o", color="tab:red")
ax2.set_ylabel("Sampled silhouette", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")

plt.title("K-Means model selection diagnostics")
plt.show()

## Cluster Stability

K-Means can change with initialization. The adjusted Rand index compares labels from several random seeds against the default seed. Values closer to 1 indicate more stable assignments.

In [ ]:
seed_values = [0, 7, 21, 42, 99]
base_labels = KMeans(n_clusters=OPTIMAL_K, init="k-means++", random_state=RANDOM_STATE, n_init=10).fit_predict(df_scaled)
stability_rows = []

for seed in seed_values:
    labels = KMeans(n_clusters=OPTIMAL_K, init="k-means++", random_state=seed, n_init=10).fit_predict(df_scaled)
    counts = pd.Series(labels).value_counts()
    stability_rows.append(
        {
            "seed": seed,
            "adjusted_rand_vs_seed_42": adjusted_rand_score(base_labels, labels),
            "min_cluster_size": int(counts.min()),
            "max_cluster_size": int(counts.max()),
        }
    )

stability_df = pd.DataFrame(stability_rows)
display(stability_df)

## Fit Final Model And Write Output

The final model uses k=4 on the full scaled feature set. Labels are assigned to a separate output dataframe with matching indexes, avoiding accidental misalignment with filtered rows.

In [ ]:
final_kmeans = KMeans(n_clusters=OPTIMAL_K, init="k-means++", random_state=RANDOM_STATE, n_init=10)
cluster_labels = final_kmeans.fit_predict(df_scaled)

output_columns = [
    "name",
    "category_list",
    "market",
    "funding_total_usd",
    "status",
    "country_code",
    "city",
    "funding_rounds",
    "founded_at",
    "first_funding_at",
    "last_funding_at",
    "venture",
    "debt_financing",
    "private_equity",
    "stage_level",
    "early_stage_funding",
    "early_stage_funding_ratio",
    "venture_ratio",
    "debt_financing_ratio",
    "private_equity_ratio",
    "log_funding_total_usd",
    "log_funding_rounds",
    "log_early_stage_funding",
    "log_venture",
    "log_debt_financing",
    "log_private_equity",
]
output_columns = [col for col in output_columns if col in df_clustering.columns]

df_output = df_clustering.loc[df_scaled.index, output_columns].copy()
df_output["Cluster"] = pd.Series(cluster_labels, index=df_scaled.index).astype(int)

assert df_output.index.equals(df_scaled.index)
assert df_output["Cluster"].nunique() == OPTIMAL_K
assert not df_output["Cluster"].isna().any()

df_output.to_csv(OUTPUT_DATA, index=False)

cluster_counts = (
    df_output["Cluster"]
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)
cluster_counts["percentage"] = cluster_counts["count"] / len(df_output) * 100

display(cluster_counts)
print(f"Saved {len(df_output):,} rows to {OUTPUT_DATA}")

## PCA Visualization

PCA is used only to inspect the final clusters visually. The final K-Means labels still come from the full scaled feature matrix above.

In [ ]:
pca_full = PCA().fit(df_scaled)
pca_summary = pd.DataFrame(
    {
        "component": np.arange(1, len(pca_full.explained_variance_ratio_) + 1),
        "explained_variance_ratio": pca_full.explained_variance_ratio_,
        "cumulative_variance": np.cumsum(pca_full.explained_variance_ratio_),
    }
)
display(pca_summary.head(10))

pca_vis = PCA(n_components=3)
pca_components = pca_vis.fit_transform(df_scaled)
df_pca = pd.DataFrame(pca_components, columns=["PC1", "PC2", "PC3"], index=df_scaled.index)
df_pca["Cluster"] = df_output["Cluster"].astype(str).values
df_pca["name"] = df_output["name"].values
df_pca["market"] = df_output["market"].values

print(f"Variance explained by the first 3 PCs: {pca_vis.explained_variance_ratio_.sum() * 100:.2f}%")

fig = px.scatter_3d(
    df_pca,
    x="PC1",
    y="PC2",
    z="PC3",
    color="Cluster",
    hover_name="name",
    hover_data={"market": True},
    title="Startup funding clusters visualized with PCA",
    opacity=0.7,
)
fig.update_layout(margin=dict(l=0, r=0, b=0, t=40))
fig.show()

## Modeling Limitations

This is an exploratory segmentation, not a predictive model. The dataset is sparse, funding subtype totals can disagree with total funding, and Crunchbase-style records can be incomplete or biased toward visible companies. Cluster labels are arbitrary K-Means IDs, so the interpretation notebook assigns names only after profiling the saved clusters.